In [4]:
!pip install monai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 23.1 MB/s eta 0:00:0000:0100:01


In [5]:
import os
import numpy as np
import pandas as pd
import pydicom
import torch
from tqdm.auto import tqdm
from monai.transforms import (
    Compose, LoadImaged, Spacingd, Orientationd, EnsureChannelFirstd,
    ScaleIntensityRanged, Resized, CopyItemsd, ConcatItemsd, DeleteItemsd, MapTransform,
    CropForegroundd
)
from monai.data import MetaTensor
import shutil

class SelectiveSamplingd(MapTransform):
    def __init__(self, keys, num_slices=64):
        super().__init__(keys)
        self.num_slices = num_slices

    def __call__(self, data):
        d = dict(data)
        for key in self.keys:
            img = d[key] # (C, S, H, W)
            s = img.shape[1]
            # 64장을 뽑을 인덱스 계산
            indices = np.linspace(0, s - 1, self.num_slices).astype(int)
            d[key] = img[:, indices, :, :]
        return d

class Preprocessor:
    def __init__(self, save_dir, target_slices=64, target_size=224):
        self.save_dir = save_dir
        self.transforms = Compose([

            # 1. 방향 통일 (코드 B의 장점: 해부학적 위치 고정)
            Orientationd(keys=["image"], axcodes="RAS"),
            
            # 2. 물리적 간격 표준화 (1.5mm Isotropic)
            Spacingd(keys=["image"], pixdim=(1.5, 1.5, 1.5), mode="bilinear"),
            
            # 3. 외상 특화 윈도잉 3채널 생성 (코드 A의 장점)
            CopyItemsd(keys=["image"], times=3, names=["img_soft", "img_angio", "img_bowel"]),
            
            # Soft Tissue (장기 손상용)
            ScaleIntensityRanged(keys=["img_soft"], a_min=-160, a_max=240, b_min=0.0, b_max=1.0, clip=True),
            # Angio/Blood (활성 출혈 점 강조용)
            ScaleIntensityRanged(keys=["img_angio"], a_min=-250, a_max=450, b_min=0.0, b_max=1.0, clip=True),
            # Bowel/Air (장 천공 가스 강조용)
            ScaleIntensityRanged(keys=["img_bowel"], a_min=-300, a_max=200, b_min=0.0, b_max=1.0, clip=True),
            
            # 4. 채널 결합 및 불필요 항목 삭제
            ConcatItemsd(keys=["img_soft", "img_angio", "img_bowel"], name="image"),
            DeleteItemsd(keys=["img_soft", "img_angio", "img_bowel"]),
            
            # 5. 크기 표준화 (깊이 64, 가로세로 128로 리사이징)
            # 검은 공기(배경) 잘라내기 (환자 몸통만 남김)
            # 윈도잉으로 공기를 0으로 만든 직후에 써야 합니다.
            # margin=5를 주어 피부 바깥쪽 정보가 너무 칼같이 잘리지 않게 보호합니다.
            CropForegroundd(keys=["image"], source_key="image", margin=5),
            
            # 64장 샘플링 (위의 SelectiveSamplingd 사용)
            SelectiveSamplingd(keys=["image"], num_slices=64),
            
            # 5. 가로세로만 리사이즈 (-1은 깊이(64장)를 유지하라는 뜻)
            Resized(keys=["image"], spatial_size=(-1, target_size, target_size))
        ])

    def get_valid_dicom_files(self, dcm_dir):
        """코드 A의 장점: 손상된 DICOM 파일을 사전에 걸러냄"""
        valid_slices = []

        for f in os.listdir(dcm_dir):
            path = os.path.join(dcm_dir, f)
            try:
                ds = pydicom.dcmread(path)
                _ = ds.pixel_array # RLE 디코딩 테스트
                
                if "ImagePositionPatient" in ds:
                    valid_slices.append(ds)
            except:
                continue
                
        if len(valid_slices) < 10:
            return None
        
        # Z-Coordinate 기준 정렬 (해부학적 순서)
        valid_slices.sort(key=lambda x: float(x.ImagePositionPatient[2]))

        # HU 변환 및 볼륨 생성
        vol = []
        for s in valid_slices:
            img = s.pixel_array.astype(np.float32)
            slope = valid_slices[0].RescaleSlope
            intercept = valid_slices[0].RescaleIntercept
            vol.append(img * slope + intercept)
            
        vol = np.stack(vol, axis=0)
        
        if len(valid_slices) > 1:
            z_spacing = np.abs(valid_slices[1].ImagePositionPatient[2] - valid_slices[0].ImagePositionPatient[2]) 
        else:
            z_spacing = valid_slices[0].SliceThickness
            
        curr_spacing = (
            z_spacing, 
            float(valid_slices[0].PixelSpacing[0]), 
            float(valid_slices[0].PixelSpacing[1])
        )
        
        # 2. 현재 Spacing을 기반으로 Affine 행렬(4x4) 생성
        # 1.0은 monai에서 필요한 4x4를 맞추기 위해, 원래는 3x3
        affine = np.diag(list(curr_spacing) + [1.0])

        # 3. MetaTensor 생성 (데이터 + Affine을 하나로 묶음)
        # vol: (D, H, W) -> [None] 추가하여 (C, D, H, W) 형태로 변환
        vol_meta = MetaTensor(vol[None], affine=affine)
        
        return vol_meta

    def run(self, data_list):
        
        for item in tqdm(data_list, desc="전처리 진행 중"):
            patient_id = str(item['PatientID'])
            series_path = item['series_path']
            series_id = series_path.split('/')[-1]
            
            full_dcm_path = os.path.join(BASE_DIR, series_path)
            
            vol_meta = self.get_valid_dicom_files(full_dcm_path)
                
            try:
                # MONAI 파이프라인 실행
                # LoadImaged는 파일 리스트를 직접 받을 수 있습니다.
                data = self.transforms({"image": vol_meta})
                
                # 결과물 추출 (C, D, H, W) -> (3, 64, 128, 128)
                final_volume = data["image"].numpy()
                
                # 저장 (환자 ID별 폴더 생성)
                out_path = os.path.join(self.save_dir, f"{series_id}.npy")
                
                # 용량 절약을 위해 float16 저장
                np.save(out_path, final_volume.astype(np.float16))
                
            except Exception as e:
                print(f"Error processing {series_path}: {e}")



/usr/local/lib/python3.12/dist-packages/sqlalchemy/orm/query.py:195: SyntaxWarning: "is not" with 'tuple' literal. Did you mean "!="?
  if entities is not ():
2026-01-22 13:04:24.331677: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769087064.659942      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769087064.748648      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769087065.542836      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769087065.542902      55 computation_placer.cc:177] computation placer already registered.

In [ ]:
BASE_DIR = '/kaggle/input/rsna-2023-abdominal-trauma-detection/'

# 전처리된 데이터를 저장할 폴더
SAVE_DIR = '/kaggle/working/processed_files'
os.makedirs(SAVE_DIR, exist_ok=True)

# 파일 읽기
train_df = pd.read_csv(f'{BASE_DIR}train_2024.csv') # 파일명 확인 필요
tags_df = pd.read_parquet(f'{BASE_DIR}train_dicom_tags.parquet')

injury_columns = [
    'bowel_injury', 
    'extravasation_injury', 
    'kidney_low', 'kidney_high', 
    'liver_low', 'liver_high', 
    'spleen_low', 'spleen_high'
]

print("--- 데이터 무결성 검사 시작 ---")
for col in injury_columns:
    # any_injury가 0인데 개별 장기 부상이 1인 경우 (논리적 모순)
    bad_data = train_df[(train_df['any_injury'] == 0) & (train_df[col] == 1)]
    
    if len(bad_data) > 0:
        print(f"[경고] {col} 컬럼에 모순 발견: {len(bad_data)}건")
        print(f"해당 patient_id: {bad_data['patient_id'].tolist()}")
    else:
        print(f"[정상] {col} 결함 없음")

# 반대의 경우도 체크: any_injury가 1인데 모든 장기가 Healthy인 경우
# 모든 부상 컬럼의 합이 0인데 any_injury만 1인 경우
no_injury_but_flagged = train_df[(train_df['any_injury'] == 1) & (train_df[injury_columns].sum(axis=1) == 0)]
if len(no_injury_but_flagged) > 0:
    print(f"\n[경고] 부상 부위가 없는데 any_injury만 1인 데이터: {len(no_injury_but_flagged)}건")

# 고유 폴더 경로 추출 및 환자 ID 연결
tags_df['series_path'] = tags_df['path'].str.split('/').str[:-1].str.join('/')
unique_series = tags_df[['PatientID', 'series_path']].drop_duplicates()


unique_series = unique_series[:1000]
unique_series_list = unique_series.to_dict('records') # 이 줄을 추가하세요
total_len = len(unique_series)

preprocessor = Preprocessor(SAVE_DIR)
preprocessor.run(unique_series_list)
 
print("최종 압축 시작...")
# processed_files 폴더만 압축하여 'results.zip' 생성
shutil.make_archive('/kaggle/working/final_output', 'zip', SAVE_DIR)
print("압축 완료!")

--- 데이터 무결성 검사 시작 ---
[정상] bowel_injury 결함 없음
[정상] extravasation_injury 결함 없음
[정상] kidney_low 결함 없음
[정상] kidney_high 결함 없음
[정상] liver_low 결함 없음
[정상] liver_high 결함 없음
[정상] spleen_low 결함 없음
[정상] spleen_high 결함 없음


/usr/local/lib/python3.12/dist-packages/monai/utils/deprecate_utils.py:321: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


전처리 진행 중:   0%|          | 0/711 [00:00<?, ?it/s]